# AgriYield: Data Science-Based Crop Yield Analysis and Prediction
**An Empirical Machine Learning Investigation into Agronomic Drivers of Agricultural Productivity**

This notebook presents the complete, reproducible data science workflow for the **AgriYield** project.
Dataset Source: Directorate of Economics and Statistics (DES), Ministry of Agriculture and Farmers Welfare & India Meteorological Department (IMD).


In [ ]:
# 1. Imports and Environment Setup
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="crest")
plt.rcParams["figure.figsize"] = (10, 5)
print("Environment initialized successfully!")


## 1. Data Collection & Initial Audit
We ingest the authenticated Agricultural Crop Yield in Indian States dataset (19,689 records).


In [ ]:
dataset_path = os.path.join("..", "data", "dataset.csv")
df = pd.read_csv(dataset_path)
print("Dataset Shape:", df.shape)
display(df.head())
display(df.info())
display(df.describe().T)


## 2. Data Cleaning & Domain Standardization
- Strip trailing whitespace from strings
- Verify non-negative physical bounds
- Identify Coconut unit disparity (nuts/ha vs metric tonnes/ha)


In [ ]:
df.columns = [col.strip() for col in df.columns]
for col in ["Crop", "Season", "State"]:
    df[col] = df[col].astype(str).str.strip()

df["Yield_Unit"] = np.where(df["Crop"] == "Coconut", "Nuts/Ha", "Tonnes/Ha")
print("Zero Yield observations:", (df["Yield"] == 0).sum())
print("Cleaned unique crops:", df["Crop"].nunique())


## 3. Exploratory Data Analysis (EDA)
We investigate yield distributions, regional variations, and environmental drivers.


In [ ]:
field_df = df[df["Crop"] != "Coconut"]
plt.figure(figsize=(10, 4))
sns.histplot(field_df["Yield"], bins=50, kde=True, color="teal")
plt.title("Figure 4: Crop Yield Distribution (Tonnes/Ha)")
plt.xlabel("Yield (Tonnes/Ha)")
plt.xlim(0, 80)
plt.show()


In [ ]:
top_crops = field_df.groupby("Crop")["Yield"].mean().sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 5))
sns.barplot(x=top_crops.values, y=top_crops.index, palette="viridis")
plt.title("Figure 5: Mean Crop Yield by Species (Tonnes/Ha)")
plt.xlabel("Mean Yield (t/ha)")
plt.show()


In [ ]:
sample = field_df.sample(2000, random_state=42)
plt.figure(figsize=(8, 5))
sns.scatterplot(data=sample, x="Annual_Rainfall", y="Yield", alpha=0.5, hue="Season")
plt.yscale("log")
plt.title("Figure 8: Annual Rainfall vs. Crop Yield (Log Scale)")
plt.xlabel("Annual Rainfall (mm)")
plt.ylabel("Yield (t/ha) [Log Scale]")
plt.show()


## 4. Statistical Analysis & Hypothesis Testing
We perform Pearson & Spearman correlation testing, Kruskal-Wallis variance tests, and check Simpson's Paradox.


In [ ]:
groups = [group["Yield"].values for name, group in field_df.groupby("Crop")]
h_stat, p_val = stats.kruskal(*groups)
print(f"Kruskal-Wallis across Crops: H-stat = {h_stat:.2f}, p-value = {p_val:.2e}")

field_df = field_df.copy()
field_df["Fertilizer_per_Area"] = field_df["Fertilizer"] / field_df["Area"]
field_df["Pesticide_per_Area"] = field_df["Pesticide"] / field_df["Area"]
corr = field_df[["Yield", "Annual_Rainfall", "Area", "Fertilizer_per_Area", "Pesticide_per_Area"]].corr(method="spearman")
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Spearman Correlation Matrix")
plt.show()


## 5. Feature Engineering & Time-Aware Train/Test Split
- Strictly exclude `Production` to prevent data leakage
- Train on historical harvests (1997-2015), test on future harvests (2016-2020)


In [ ]:
sys.path.append(os.path.join("..", "src"))
from feature_engineering import split_data, build_preprocessor

X_train, X_test, y_train, y_test, info = split_data(df, split_method="time_aware", test_year_cutoff=2016)
print("Train samples (1997-2015):", len(X_train))
print("Test samples (2016-2020):", len(X_test))


## 6. Machine Learning Model Training & Evaluation
We evaluate 4 candidate regressors under target log-transformation to prevent negative yield outputs.


In [ ]:
from evaluate_models import load_evaluation_data
comp_data, test_preds = load_evaluation_data()
comp_df = pd.DataFrame(comp_data["comparison"])
display(comp_df)


In [ ]:
rf_preds = test_preds["Pred_Random Forest"]
actual = test_preds["Actual_Yield"]
plt.figure(figsize=(7, 7))
plt.scatter(actual, rf_preds, alpha=0.4, color="darkcyan")
plt.plot([0.1, 1000], [0.1, 1000], color="red", linestyle="--", label="Ideal (y=x)")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("Actual Yield (Tonnes/Ha)"); plt.ylabel("Predicted Yield (Tonnes/Ha)")
plt.title("Figure 11: Actual vs. Predicted Yield — Random Forest")
plt.legend()
plt.show()


## 7. Model Interpretation: Feature Importance
Decomposing predictive drivers to examine agronomic contribution.


In [ ]:
feat_imp_path = os.path.join("..", "models", "feature_importance.json")
with open(feat_imp_path) as f:
    feat_data = json.load(f)
grouped = feat_data["Random Forest"]["grouped_feature_importance_pct"]
plt.figure(figsize=(9, 4))
sns.barplot(x=list(grouped.values()), y=list(grouped.keys()), palette="mako")
plt.title("Figure 12: Grouped Feature Importance in Predicting Crop Yield (%)")
plt.xlabel("Relative Gini Importance (%)")
plt.show()


## 8. Real-Time Inference Demo
Simulating pre-harvest yield prediction for Wheat in Punjab.


In [ ]:
from prediction import predict_crop_yield
result = predict_crop_yield(
    crop="Wheat", state="Punjab", season="Rabi", crop_year=2024,
    area=100.0, annual_rainfall=650.0, fertilizer=15000.0, pesticide=250.0
)
for k, v in result.items():
    if k != "input_summary":
        print(f"{k}: {v}")
